In [2]:
!pip uninstall -y torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv
!pip install torch-geometric


Found existing installation: torch-geometric 2.6.1
Uninstalling torch-geometric-2.6.1:
  Successfully uninstalled torch-geometric-2.6.1


Defaulting to user installation because normal site-packages is not writeable
  Using cached torch_geometric-2.6.1-py3-none-any.whl.metadata (63 kB)
Using cached torch_geometric-2.6.1-py3-none-any.whl (1.1 MB)


In [2]:
import numpy as np
import pandas as pd
import random
from torch_geometric.data import HeteroData
import torch


In [3]:
user_df = pd.read_csv('../../data/preproccedData/Augmented_PreProccedNutrationParameters.csv')
food_df = pd.read_csv('../../data/RecommandationDatasets/NutritionDatasets/Foods_Datasets.csv')
disease_df = pd.read_csv('../../data/RecommandationDatasets/NutritionDatasets/diseases.csv')
nutrition_df = pd.read_csv('../../data/RecommandationDatasets/NutritionDatasets/nutrients.csv')

In [4]:
user_df.head()

,Age,Gender,Height,Weight,Carbohydrate_Consumption,Protein_Intake,Fat_Intake,Regularity_of_Meals,Portion_Control,Hydration,Caloric_Balance,Sugar_Consumption,BMI,DiabetesRisk,NutritionRisk
0,24,1,160.39,53.08,2,1,1,1,1,2,2106.59,1,16.60,29.69,54.09
1,24,1,158.47,58.42,1,2,1,0,1,1,2106.58,1,18.71,31.04,36.07
2,28,0,146.93,50.97,3,3,1,0,2,2,2106.58,1,18.99,28.55,18.03
3,24,0,156.55,59.46,2,1,1,1,3,5,1378.17,2,19.52,36.28,36.04
4,22,0,147.91,48.84,2,1,1,1,2,3,2106.59,3,17.95,24.35,18.03


In [5]:
user_df["user_id"] = ["u" + str(i + 1) for i in range(len(user_df))]

# Randomly assign preferred culture
cultures = ["Sri Lankan", "Western", "Mixed"]
user_df["preferred_culture"] = [random.choice(cultures) for _ in range(len(user_df))]

# Save the required columns
users_gnn = user_df[["user_id", "preferred_culture"]]

In [6]:
user_df.head()

,Age,Gender,Height,Weight,Carbohydrate_Consumption,Protein_Intake,Fat_Intake,Regularity_of_Meals,Portion_Control,Hydration,Caloric_Balance,Sugar_Consumption,BMI,DiabetesRisk,NutritionRisk,user_id,preferred_culture
0,24,1,160.39,53.08,2,1,1,1,1,2,2106.59,1,16.60,29.69,54.09,u1,Sri Lankan
1,24,1,158.47,58.42,1,2,1,0,1,1,2106.58,1,18.71,31.04,36.07,u2,Western
2,28,0,146.93,50.97,3,3,1,0,2,2,2106.58,1,18.99,28.55,18.03,u3,Mixed
3,24,0,156.55,59.46,2,1,1,1,3,5,1378.17,2,19.52,36.28,36.04,u4,Western
4,22,0,147.91,48.84,2,1,1,1,2,3,2106.59,3,17.95,24.35,18.03,u5,Sri Lankan


In [7]:
user_df.to_csv("../../data/RecommandationDatasets/NutritionDatasets/Updated_User_Nutrition_Parameters.csv", index=False)

In [8]:
food_df.head()

,food_id,food_item,calories,carbs,protein,fat,glycemic_index,meal_type,culture,estimated_weight_g
0,f1000,Millet with fish and tomato,1140.535373,152.9,45.9,33.6,52,Breakfast,Sri Lankan,877
1,f1001,Rice with chicken and cabbage,750.478011,78.3,39.0,29.0,69,Breakfast,Sri Lankan,577
2,f1002,Millet with chicken and beans,1197.418738,149.6,52.2,40.9,52,Dinner,Sri Lankan,921
3,f1003,Millet with beef and onion,1271.988528,72.4,35.8,92.9,63,Lunch,Sri Lankan,978
4,f1004,Wheat with egg and cucumber,948.852773,120.8,37.1,32.2,69,Dinner,Sri Lankan,730


In [9]:
num_foods = len(food_df)
food_df['food_id'] = [f'f{i:04d}' for i in range(1, num_foods + 1)]
food_df['calories'] = food_df['calories'].round(1)

# Save the required columns
foods_gnn = food_df[['food_id', 'culture']]

# Display the updated DataFrame
food_df.head()

,food_id,food_item,calories,carbs,protein,fat,glycemic_index,meal_type,culture,estimated_weight_g
0,f0001,Millet with fish and tomato,1140.5,152.9,45.9,33.6,52,Breakfast,Sri Lankan,877
1,f0002,Rice with chicken and cabbage,750.5,78.3,39.0,29.0,69,Breakfast,Sri Lankan,577
2,f0003,Millet with chicken and beans,1197.4,149.6,52.2,40.9,52,Dinner,Sri Lankan,921
3,f0004,Millet with beef and onion,1272.0,72.4,35.8,92.9,63,Lunch,Sri Lankan,978
4,f0005,Wheat with egg and cucumber,948.9,120.8,37.1,32.2,69,Dinner,Sri Lankan,730


In [10]:


# 1. User hasRisk Diabetes (for demo, assume all users have diabetes risk)
user_risk_edges = user_df['user_id'].apply(lambda uid: [uid, 'hasRisk', 'd1']).tolist()

# 2. User prefers culture
user_pref_edges = user_df[['user_id', 'preferred_culture']].apply(lambda row: [row['user_id'], 'prefers', row['preferred_culture']], axis=1).tolist()

# 3. Food contains nutrient
nutrient_map = {'protein': 'n1', 'carbs': 'n2', 'fat': 'n3'}
food_nutrient_edges = []
for _, row in food_df.iterrows():
    fid = row['food_id']
    for col, nid in nutrient_map.items():
        if col in row and not pd.isna(row[col]) and float(row[col]) > 0:
            food_nutrient_edges.append([fid, 'contains', nid])

# 4. Food raises glycemic index category
def gi_category(gi):
    try:
        gi = float(gi)
        if gi < 55: return "Low_GI"
        elif gi <= 69: return "Medium_GI"
        else: return "High_GI"
    except:
        return None

food_gi_edges = []
for _, row in food_df.iterrows():
    category = gi_category(row['glycemic_index'])
    if category:
        food_gi_edges.append([row['food_id'], 'raises', category])

# Combine all edges
all_edges = user_risk_edges + user_pref_edges + food_nutrient_edges + food_gi_edges
edges_df = pd.DataFrame(all_edges, columns=['source', 'relation', 'target'])

# Save to CSV
edges_path = "../../data/RecommandationDatasets/NutritionDatasets/edges.csv"
edges_df.to_csv(edges_path, index=False)

edges_path


'../../data/RecommandationDatasets/NutritionDatasets/edges.csv'

In [11]:

from collections import defaultdict

# Load datasets
user_df = pd.read_csv("../../data/RecommandationDatasets/NutritionDatasets/Updated_User_Nutrition_Parameters.csv")
food_df = pd.read_csv("../../data/RecommandationDatasets/NutritionDatasets/Foods_Datasets.csv")
nutrition_df = pd.read_csv("../../data/RecommandationDatasets/NutritionDatasets/nutrients.csv")
disease_df = pd.read_csv("../../data/RecommandationDatasets/NutritionDatasets/diseases.csv")
edges = pd.read_csv("../../data/RecommandationDatasets/NutritionDatasets/edges.csv")

# Encode categorical columns
risk_map = {'Low': 0.0, 'Medium': 1.0, 'High': 2.0}
user_df['DiabetesRisk'] = user_df['DiabetesRisk'].map(risk_map)
user_df['NutritionRisk'] = user_df['NutritionRisk'].map(risk_map)

# Create HeteroData graph
data = HeteroData()

# Add user features
user_features = user_df[['Age', 'Gender', 'Height', 'Weight', 'Carbohydrate_Consumption',
                         'Protein_Intake', 'Fat_Intake', 'Regularity_of_Meals', 'Portion_Control',
                         'Caloric_Balance', 'Sugar_Consumption', 'BMI',
                         'DiabetesRisk', 'NutritionRisk']].astype(float)
data['user'].x = torch.tensor(user_features.values, dtype=torch.float)
user_id_map = {uid: i for i, uid in enumerate(user_df['user_id'])}

# Add food features (no food_id in feature set!)
food_features = food_df[['calories', 'carbs', 'protein', 'fat', 'glycemic_index', 'estimated_weight_g']].astype(float)
data['food'].x = torch.tensor(food_features.values, dtype=torch.float)
food_id_map = {fid: i for i, fid in enumerate(food_df['food_id'])}

# Add nutrient and disease nodes
data['nutrient'].x = torch.eye(len(nutrition_df))
data['disease'].x = torch.eye(len(disease_df))

# Add edges
edge_index_dict = defaultdict(list)
for _, row in edges.iterrows():
    src, rel, tgt = row['source'], row['relation'], row['target']
    
    if rel == "hasRisk" and src in user_id_map:
        edge_index_dict[('user', 'hasRisk', 'disease')].append([user_id_map[src], 0])
    
    elif rel == "contains" and src in food_id_map:
        edge_index_dict[('food', 'contains', 'nutrient')].append([food_id_map[src], int(tgt[1:]) - 1])

# Convert edges to tensors
for edge_type, edge_list in edge_index_dict.items():
    edge_tensor = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    data[edge_type].edge_index = edge_tensor


In [12]:

from torch.nn import Linear
from torch_geometric.nn import GATConv, HeteroConv
import torch.nn.functional as F

class GNN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = HeteroConv({
    ('user', 'hasRisk', 'disease'): GATConv((-1, -1), hidden_channels, add_self_loops=False),
    ('food', 'contains', 'nutrient'): GATConv((-1, -1), hidden_channels, add_self_loops=False),
    ('disease', 'rev_hasRisk', 'user'): GATConv((-1, -1), hidden_channels, add_self_loops=False),
    ('nutrient', 'rev_contains', 'food'): GATConv((-1, -1), hidden_channels, add_self_loops=False),
}, aggr='sum')


        self.lin = Linear(hidden_channels, hidden_channels)

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {k: F.relu(v) for k, v in x_dict.items()}
        x_dict = {k: self.lin(v) for k, v in x_dict.items()}
        return x_dict



In [13]:
# Add reverse edges for message passing INTO 'user' and 'food'
if ('user', 'hasRisk', 'disease') in data.edge_index_dict:
    edge = data['user', 'hasRisk', 'disease'].edge_index
    data['disease', 'rev_hasRisk', 'user'].edge_index = edge.flip(0)

if ('food', 'contains', 'nutrient') in data.edge_index_dict:
    edge = data['food', 'contains', 'nutrient'].edge_index
    data['nutrient', 'rev_contains', 'food'].edge_index = edge.flip(0)


In [14]:
model = GNN(hidden_channels=64)  # or another number depending on your setup
model.eval()

with torch.no_grad():
    out = model(data.x_dict, data.edge_index_dict)
user_emb = out['user']   # [num_users, hidden_dim]
food_emb = out['food']   # [num_foods, hidden_dim]



In [15]:
# Dot product similarity between each user and each food
scores = torch.matmul(user_emb, food_emb.T)  # Shape: [num_users, num_foods]


In [16]:
top_k = 5  # Number of foods to recommend per user
top_scores, top_indices = torch.topk(scores, k=top_k, dim=1)


In [17]:
# If you used this earlier:
# food_id_map = {food_id_string: index}
index_to_food_id = {idx: fid for fid, idx in food_id_map.items()}

# Example: recommendations for first user
print("Top recommendations for User 0:")
for food_idx in top_indices[0]:
    print(index_to_food_id[int(food_idx)])


Top recommendations for User 0:
f1003
f1004
f1001
f1000
f1002


In [18]:
print(data)


HeteroData(
  user={ x=[1062, 14] },
  food={ x=[2500, 6] },
  nutrient={ x=[4, 4] },
  disease={ x=[1, 1] },
  (user, hasRisk, disease)={ edge_index=[2, 1062] },
  (food, contains, nutrient)={ edge_index=[2, 4503] },
  (disease, rev_hasRisk, user)={ edge_index=[2, 1062] },
  (nutrient, rev_contains, food)={ edge_index=[2, 4503] }
)


In [19]:
print(data['user', 'hasRisk', 'disease'].edge_index.shape)
print(data['food', 'contains', 'nutrient'].edge_index.shape)


torch.Size([2, 1062])
torch.Size([2, 4503])


In [20]:
print(data['user', 'hasRisk', 'disease'].edge_index.T[:5])  # First 5 edges

tensor([[0, 0],
        [1, 0],
        [2, 0],
        [3, 0],
        [4, 0]])


In [35]:
import pandas as pd
import random
import json

# 🧑‍⚕️ User Profile with Risk Factors
test_user = {
    "user_id": "u101",
    "Age": 45,
    "Gender": "Female",
    "BMI": 28.0,                # High BMI (Overweight)
    "DiabetesRisk": 0.8,        # High Diabetes Risk
    "Preferences": "Sri Lankan"
}

# 🍱 Categories
categories = ["Breakfast", "Lunch", "Dinner", "Snack"]

# ✅ Suitability with Risk Reduction
def is_suitable_risk_aware(food, user):
    # Diabetic filtering
    if user["DiabetesRisk"] > 0.6:
        if food["glycemic_index"] > 55 or food["carbs"] > 60:
            return False
    
    # Obesity filtering
    if user["BMI"] > 25:
        if food["calories"] > 700 or food["fat"] > 25:
            return False

    # Overeating portion filtering
    if food["estimated_weight_g"] > 800:
        return False

    # Cultural match
    if food["culture"] != user["Preferences"]:
        return False

    return True

# 🧠 Plan Generator
def generate_health_aware_meal_plan(user, food_df):
    used_food_ids = set()
    meal_plan = {}

    for day in range(1, 8):
        daily_plan = {}
        for meal in categories:
            options = food_df[
                (food_df["meal_type"] == meal) &
                (~food_df["food_id"].isin(used_food_ids))
            ].copy()

            options = options[options.apply(lambda row: is_suitable_risk_aware(row, user), axis=1)]

            if options.empty:
                daily_plan[meal] = {
                    "food": "No suitable meal",
                    "estimated_weight_g": 0,
                    "calories": 0,
                    "carbs": 0,
                    "protein": 0,
                    "fat": 0,
                    "glycemic_index": "-"
                }
            else:
                chosen = options.sample(1).iloc[0]
                used_food_ids.add(str(chosen["food_id"]))
                daily_plan[meal] = {
                    "food": str(chosen["food_item"]),
                    "estimated_weight_g": int(chosen["estimated_weight_g"]),
                    "calories": float(chosen["calories"]),
                    "carbs": float(chosen["carbs"]),
                    "protein": float(chosen["protein"]),
                    "fat": float(chosen["fat"]),
                    "glycemic_index": int(chosen["glycemic_index"])
                }

        meal_plan[f"Day {day}"] = daily_plan

    return meal_plan

# 🛠️ Run the plan generator
meal_plan = generate_health_aware_meal_plan(test_user, food_df)

# 📤 Output
print(json.dumps(meal_plan, indent=2))


{
  "Day 1": {
    "Breakfast": {
      "food": "Kurakkan with beef and cabbage",
      "estimated_weight_g": 258,
      "calories": 335.8030593,
      "carbs": 5.8,
      "protein": 54.6,
      "fat": 9.3,
      "glycemic_index": 52
    },
    "Lunch": {
      "food": "Bread with tofu and cabbage",
      "estimated_weight_g": 368,
      "calories": 478.0114723,
      "carbs": 53.5,
      "protein": 22.4,
      "fat": 15.7,
      "glycemic_index": 46
    },
    "Dinner": {
      "food": "Bread with tofu and cucumber",
      "estimated_weight_g": 403,
      "calories": 523.9005736,
      "carbs": 32.5,
      "protein": 43.1,
      "fat": 22.2,
      "glycemic_index": 54
    },
    "Snack": {
      "food": "Kurakkan with mung beans and carrot",
      "estimated_weight_g": 269,
      "calories": 350.1434034,
      "carbs": 12.1,
      "protein": 27.6,
      "fat": 20.2,
      "glycemic_index": 54
    }
  },
  "Day 2": {
    "Breakfast": {
      "food": "Kurakkan with lentils and beans",
 